In [2]:

import zipfile
import os

zip_path = "/content/archive (1).zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print(os.listdir(extract_path))


Dataset extracted successfully!
['1661-0.txt']


In [3]:

import glob

text_files = glob.glob(
    "/content/dataset/**/*.txt",
    recursive=True
)

print("Text files found:", text_files)

with open(text_files[0], "r", encoding="utf-8", errors="ignore") as f:
    text_data = f.read()

print("\nFirst 500 characters:")
print(text_data[:500])


Text files found: ['/content/dataset/1661-0.txt']

First 500 characters:
﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Charac


In [4]:

import re

# Remove the Project Gutenberg header
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

if start_marker in text_data:
    text_data = text_data.split(start_marker, 1)[1]

if end_marker in text_data:
    text_data = text_data.split(end_marker, 1)[0]

# Normalize whitespace
text_data = re.sub(r"\s+", " ", text_data).strip()

# Split the book into sentences
sentences = re.split(r'(?<=[.!?])\s+', text_data)

# Keep meaningful sentences
sentences = [
    s.strip().lower()
    for s in sentences
    if len(s.split()) >= 3
]

# Select approximately 10,000 sentences
sentences = sentences[:10000]

print("Total sentences:", len(sentences))
print("\nFirst 5 sentences:")

for sentence in sentences[:5]:
    print(sentence)


Total sentences: 5004

First 5 sentences:
﻿ project gutenberg's the adventures of sherlock holmes, by arthur conan doyle this ebook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever.
you may copy it, give it away or re-use it under the terms of the project gutenberg license included with this ebook or online at www.gutenberg.net title: the adventures of sherlock holmes author: arthur conan doyle release date: november 29, 2002 [ebook #1661] last updated: may 20, 2019 language: english character set encoding: utf-8 *** start of this project gutenberg ebook the adventures of sherlock holmes *** produced by an anonymous project gutenberg volunteer and jose menendez cover the adventures of sherlock holmes by arthur conan doyle contents i.
a scandal in bohemia ii.
the red-headed league iii.
a case of identity iv.


In [5]:

import re

# Reload the original book text
with open(
    "/content/dataset/1661-0.txt",
    "r",
    encoding="utf-8-sig",
    errors="ignore"
) as f:
    raw_text = f.read()

# Remove Gutenberg header and footer
start = re.search(
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
    raw_text,
    flags=re.IGNORECASE | re.DOTALL
)

end = re.search(
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*",
    raw_text,
    flags=re.IGNORECASE | re.DOTALL
)

if start and end:
    raw_text = raw_text[start.end():end.start()]

# Remove the table of contents
raw_text = re.sub(
    r"(?is)\bcontents\b.*?(?=\bI\.\s+A SCANDAL IN BOHEMIA\b)",
    "",
    raw_text,
    count=1
)

# Normalize whitespace
raw_text = re.sub(r"\s+", " ", raw_text).strip()

# Split into sentences
sentences = re.split(r'(?<=[.!?])\s+', raw_text)

sentences = [
    s.strip().lower()
    for s in sentences
    if len(s.split()) >= 3
]

# Use all available sentences, up to 10,000
sentences = sentences[:10000]

print("Total cleaned sentences:", len(sentences))
print("\nFirst 5 cleaned sentences:")
for s in sentences[:5]:
    print(s)



Total cleaned sentences: 4903

First 5 cleaned sentences:
produced by an anonymous project gutenberg volunteer and jose menendez cover the adventures of sherlock holmes by arthur conan doyle i.
a scandal in bohemia ii.
the red-headed league iii.
a case of identity iv.
the boscombe valley mystery v.


In [6]:

import re

with open(
    "/content/dataset/1661-0.txt",
    "r",
    encoding="utf-8-sig",
    errors="ignore"
) as f:
    raw_text = f.read()

# Find the beginning of the first story
match = re.search(
    r"A SCANDAL IN BOHEMIA",
    raw_text,
    flags=re.IGNORECASE
)

if match:
    raw_text = raw_text[match.start():]

# Remove the Gutenberg footer
raw_text = re.split(
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK",
    raw_text,
    flags=re.IGNORECASE
)[0]

# Remove the table-of-contents heading if present at the beginning
raw_text = re.sub(
    r"(?is)^A SCANDAL IN BOHEMIA",
    "",
    raw_text,
    count=1
)

# Split into sentences
raw_text = re.sub(r"\s+", " ", raw_text).strip()
sentences = re.split(r'(?<=[.!?])\s+', raw_text)

sentences = [
    s.strip().lower()
    for s in sentences
    if len(s.split()) >= 3
]

sentences = sentences[:10000]

print("Total cleaned sentences:", len(sentences))
print("\nFirst 5 sentences:")
for s in sentences[:5]:
    print(s)


Total cleaned sentences: 4901

First 5 sentences:
the red-headed league iii.
a case of identity iv.
the boscombe valley mystery v.
the five orange pips vi.
the man with the twisted lip vii.


In [7]:

from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer
tokenizer = Tokenizer(
    num_words=10000,
    oov_token="<OOV>"
)

# Build vocabulary
tokenizer.fit_on_texts(sentences)

word_index = tokenizer.word_index
vocab_size = min(len(word_index) + 1, 10000)

print("Total sentences:", len(sentences))
print("Vocabulary size:", vocab_size)

print("\nFirst 20 words:")
print(list(word_index.items())[:20])

print("\nSample tokenized sentence:")
print(tokenizer.texts_to_sequences([sentences[0]])[0])


Total sentences: 4901
Vocabulary size: 8615

First 20 words:
[('<OOV>', 1), ('the', 2), ('and', 3), ('to', 4), ('of', 5), ('i', 6), ('a', 7), ('”', 8), ('in', 9), ('that', 10), ('it', 11), ('he', 12), ('was', 13), ('you', 14), ('his', 15), ('is', 16), ('my', 17), ('have', 18), ('as', 19), ('with', 20)]

Sample tokenized sentence:
[2, 241, 470, 665, 2551]


In [8]:

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Split sentences into training and testing data
train_sentences, test_sentences = train_test_split(
    sentences,
    test_size=0.20,
    random_state=42
)

# Generate sequences
def create_sequences(sentence_list):
    sequences = []

    for sentence in sentence_list:
        token_list = tokenizer.texts_to_sequences([sentence])[0]

        for i in range(1, len(token_list)):
            sequences.append(token_list[:i + 1])

    return sequences

train_sequences = create_sequences(train_sentences)
test_sequences = create_sequences(test_sentences)

# Limit sequence length to reduce memory usage
max_sequence_len = min(
    max(len(seq) for seq in train_sequences),
    30
)

# Prepare input and target
def prepare_data(sequences):
    sequences = [
        seq[-max_sequence_len:]
        for seq in sequences
        if len(seq) >= 2
    ]

    padded = pad_sequences(
        sequences,
        maxlen=max_sequence_len,
        padding="pre",
        truncating="pre"
    )

    X = padded[:, :-1]
    y = padded[:, -1]

    return X, y

X_train, y_train = prepare_data(train_sequences)
X_test, y_test = prepare_data(test_sequences)

print("Training sentences:", len(train_sentences))
print("Testing sentences:", len(test_sentences))
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


Training sentences: 3920
Testing sentences: 981
X_train shape: (82886, 29)
y_train shape: (82886,)
X_test shape: (19856, 29)
y_test shape: (19856,)


In [9]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

tf.random.set_seed(42)

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        mask_zero=True
    ),
    LSTM(64),
    Dropout(0.2),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=[
        "sparse_categorical_accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_accuracy"
        )
    ]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:

# Build the model using your input sequence shape
model.build(input_shape=(None, X_train.shape[1]))

# Display the model summary
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 29, 64)         │       551,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8615)           │       559,975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,144,359 (4.37 MB)

 Trainable params: 1,144,359 (4.37 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)


Epoch 1/10
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 73s 60ms/step - loss: 6.6825 - sparse_categorical_accuracy: 0.0529 - top5_accuracy: 0.1615 - val_loss: 6.4806 - val_sparse_categorical_accuracy: 0.0592 - val_top5_accuracy: 0.1843
Epoch 2/10
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 82s 59ms/step - loss: 6.1594 - sparse_categorical_accuracy: 0.0737 - top5_accuracy: 0.2113 - val_loss: 6.2696 - val_sparse_categorical_accuracy: 0.0855 - val_top5_accuracy: 0.2359
Epoch 3/10
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 72s 61ms/step - loss: 5.8628 - sparse_categorical_accuracy: 0.0981 - top5_accuracy: 0.2538 - val_loss: 6.1078 - val_sparse_categorical_accuracy: 0.1009 - val_top5_accuracy: 0.2670
Epoch 4/10
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 69s 59ms/step - loss: 5.6442 - sparse_categorical_accuracy: 0.1115 - top5_accuracy: 0.2784 - val_loss: 6.0248 - val_sparse_categorical_accuracy: 0.1145 - val_top5_accuracy: 0.2787
Epoch 5/10
1166/1166 ━━━━━━━━━━━━━━━━━━━━ 82s 59ms/step - loss: 5.4745 - sparse_categorical_accuracy: 0.1232 - t

In [12]:

test_results = model.evaluate(
    X_test,
    y_test,
    verbose=1,
    return_dict=True
)

print("\n--- Model Evaluation Results ---")
print(f"Test Loss: {test_results['loss']:.4f}")
print(
    f"Top-1 Accuracy: "
    f"{test_results['sparse_categorical_accuracy'] * 100:.2f}%"
)
print(
    f"Top-5 Accuracy: "
    f"{test_results['top5_accuracy'] * 100:.2f}%"
)


621/621 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - loss: 6.0505 - sparse_categorical_accuracy: 0.1340 - top5_accuracy: 0.3085

--- Model Evaluation Results ---
Test Loss: 6.0505
Top-1 Accuracy: 13.40%
Top-5 Accuracy: 30.85%


In [13]:

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_words(phrase, top_k=5):
    # Convert input phrase to lowercase word IDs
    token_list = tokenizer.texts_to_sequences(
        [phrase.lower()]
    )[0]

    if not token_list:
        print("No recognizable words:", phrase)
        return

    # Prepare input for the LSTM
    padded = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding="pre",
        truncating="pre"
    )

    # Predict probabilities
    predictions = model.predict(padded, verbose=0)[0]

    # Exclude padding and select the top five words
    predictions[0] = 0
    top_ids = np.argsort(predictions)[-top_k:][::-1]

    print("\nInput phrase:", phrase)
    print("Predicted next words:")

    for word_id in top_ids:
        word = tokenizer.index_word.get(
            int(word_id), "<unknown>"
        )
        probability = predictions[word_id] * 100
        print(f"{word}: {probability:.2f}%")

# Test five input phrases
test_phrases = [
    "i am going to",
    "sherlock holmes was",
    "the man was",
    "he looked at",
    "it was a"
]

for phrase in test_phrases:
    predict_next_words(phrase)



Input phrase: i am going to
Predicted next words:
the: 9.16%
be: 8.81%
me: 5.91%
my: 3.30%
you: 3.13%

Input phrase: sherlock holmes was
Predicted next words:
a: 13.00%
the: 5.09%
not: 2.44%
no: 2.31%
in: 2.21%

Input phrase: the man was
Predicted next words:
a: 12.77%
the: 4.52%
in: 2.91%
no: 2.16%
to: 2.07%

Input phrase: he looked at
Predicted next words:
the: 52.45%
his: 6.80%
my: 6.20%
a: 6.12%
me: 2.51%

Input phrase: it was a
Predicted next words:
little: 4.15%
very: 2.65%
man: 2.59%
few: 1.88%
small: 1.60%
